# Ablation on Kaggle — fusion (EEG + chin EMG) vs EEG only vs EMG only

Same model (PMRT v3), same folds and same settings as the fusion run `pmrt_mt3_cv5`; only one input is set to zero.

**Settings (right panel):** Accelerator = **GPU T4 x2** · Internet = **On**.
**Cell 0:** put the FILE_ID of `kaggle_upload.zip` (Google Drive, shared with "Anyone with the link").

**How to run:** `Save Version` → **Save & Run All (Commit)** — runs in the background (the browser can be closed).
No new fold is started after 9 h, so the run is saved before Kaggle's 12-h limit.
Finished folds (from Colab and from earlier Kaggle runs) are skipped; an unfinished fold resumes from its checkpoint.
For a 2nd run: **Add Input → Your Work → Notebook Output** of this notebook, then commit again.

**Outputs below:** taken from the log of the committed Kaggle run (Version 1, GPU T4 x2, 14245 s); EMG-only folds 1-4 were trained in this run.

## 0. Download the data from Google Drive

In [ ]:
# Download the data from Google Drive (shared link)
FILE_ID = "1A66BMnFC6gfmaqYA8VlOhiyUd2S2kKOK"
import os, zipfile, subprocess, sys
os.makedirs("/tmp/capdata", exist_ok=True)
if not os.path.exists("/tmp/capdata/x.npy"):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True); import gdown
    gdown.download(id=FILE_ID, output="/tmp/cap.zip", quiet=False)
    zipfile.ZipFile("/tmp/cap.zip").extractall("/tmp/capdata")
    os.remove("/tmp/cap.zip")
print(sorted(os.listdir("/tmp/capdata")))

Downloading...
From (original): https://drive.google.com/uc?id=1A66BMnFC6gfmaqYA8VlOhiyUd2S2kKOK
From (redirected): https://drive.google.com/uc?id=1A66BMnFC6gfmaqYA8VlOhiyUd2S2kKOK&confirm=t&uuid=b55cc59d-8b46-423a-9997-96f27813fa7c
To: /tmp/cap.zip


['folds_5.json', 'index.csv', 'runs', 'x.npy']


## 1. Settings

In [2]:
# Settings (identical to the fusion run pmrt_mt3_cv5)
N_FOLDS, SEQ_LEN = 5, 12
EPOCHS, PATIENCE, MIN_EPOCHS = 60, 12, 15
BATCH_SEQ, LR, WD = 24, 5e-4, 0.05
PER_SUBJECT = 1500
USE_AUG, USE_STAGE_HEAD, EMA_DECAY, SEED = True, True, 0.999, 42

TODO = [("emg", 0), ("eeg", 1), ("emg", 1), ("eeg", 2), ("emg", 2),
        ("eeg", 3), ("emg", 3), ("eeg", 4), ("emg", 4)]
START_LIMIT_H = 9.0

## 2. Setup

In [ ]:
# GPU, data location and results of previous runs
import sys, json, glob, shutil, time, importlib, numpy as np, pandas as pd, torch
from pathlib import Path
T_START = time.time()
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "- WARNING: no GPU (Settings -> Accelerator -> GPU T4)")
hits = sorted(glob.glob("/tmp/capdata/x.npy") + glob.glob("/kaggle/input/**/x.npy", recursive=True))
assert hits, "x.npy not found - run cell 0 (Drive download) first"
DATA = Path(hits[0]).parent
ROOT = "/kaggle/working"
for p in sorted(glob.glob("/tmp/capdata/runs") + glob.glob("/kaggle/input/**/runs", recursive=True)):
    shutil.copytree(p, Path(ROOT) / "runs", dirs_exist_ok=True)
print("data:", DATA)
for rn in sorted((Path(ROOT) / "runs").glob("*")):
    print(f"  {rn.name}: finished folds {sorted(int(f.parent.name[4:]) for f in rn.glob('fold*/test_pred.csv'))}")

device: cuda Tesla T4
data: /tmp/capdata
  pmrt_mt3_cv5: finished folds [0, 1, 2, 3, 4]
  pmrt_mt3_eegonly_cv5: finished folds [0, 1, 2, 3, 4]
  pmrt_mt3_emgonly_cv5: finished folds [0]


## 3. Model module

In [ ]:
%%writefile /kaggle/working/pmrt_mt3.py
# PMRT v3 module: Phy-SE-ResNet branches, cross-modal fusion, FPN, temporal context, training
import json, glob, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path

VERSION = "mt3"
REG = ["hr", "rmssd"]
DISORDERS = ["Healthy", "NFLE", "RBD", "PLM", "Insomnia", "Narcolepsy"]
STAGES = ["N1", "N2", "N3", "REM"]
IGNORE = -100

def build_cache(npz_dir, cache_dir, tmp_dir="/content/_cache_tmp"):
    import shutil
    final = Path(cache_dir)
    if (final / "index.csv").exists():
        return
    cache = Path(tmp_dir); shutil.rmtree(cache, ignore_errors=True); cache.mkdir(parents=True, exist_ok=True)
    files = sorted(glob.glob(f"{npz_dir}/*.npz"))
    metas, n = [], 0
    for f in files:
        with np.load(f) as d:
            metas.append((f, json.loads(str(d["meta"])), len(d["hr"]))); n += len(d["hr"])
    X = np.lib.format.open_memmap(cache / "x.npy", mode="w+", dtype=np.float16, shape=(n, 2, 625))
    rows, pos = [], 0
    for f, m, k in metas:
        with np.load(f) as d:
            X[pos:pos + k, 0] = d["eeg"]; X[pos:pos + k, 1] = d["emg"]
            clip = (d["eeg_clip"].astype(np.float32) > 0.01) | (d["emg_clip"].astype(np.float32) > 0.01)
            rows.append(pd.DataFrame({"record": m["record"], "group": m["group"], "age": m["age"], "sex": m["sex"],
                                      "stage": d["stage"], "t": d["t"], "hr": d["hr"], "rmssd": d["rmssd"],
                                      "clipped": clip}))
        pos += k
        print(f"cached {m['record']:8s} {pos:,}/{n:,}", flush=True)
    X.flush(); del X
    pd.concat(rows, ignore_index=True).to_csv(cache / "index.csv", index=False)
    final.mkdir(parents=True, exist_ok=True)
    shutil.copy(cache / "x.npy", final / "x.npy"); shutil.copy(cache / "index.csv", final / "index.csv")
    shutil.rmtree(cache, ignore_errors=True)

def load_data(cache_dir):
    X = np.load(Path(cache_dir) / "x.npy")
    idx = pd.read_csv(Path(cache_dir) / "index.csv")
    idx["dis"] = idx.group.map({g: i for i, g in enumerate(DISORDERS)}).fillna(IGNORE).astype(int)
    idx["stg"] = idx.stage.map({s: i for i, s in enumerate(STAGES)}).astype(int)
    return X, idx

def make_folds(idx, k=5, seed=42):
    rng = np.random.default_rng(seed); fold, start = {}, 0
    for g, recs in sorted(idx.groupby("group").record.unique().items()):
        recs = sorted(recs); rng.shuffle(recs)
        for i, r in enumerate(recs):
            fold[r] = int((start + i) % k)
        start += len(recs)
    return fold

def fold_rows(idx, fold, k_test, k=5):
    f = idx.record.map(fold).to_numpy()
    k_val = (k_test + 1) % k
    return {"train": np.where((f != k_test) & (f != k_val))[0],
            "val": np.where(f == k_val)[0], "test": np.where(f == k_test)[0]}

class SubjectBalancedBatches:
    def __init__(self, X, Y, rows, rec, batch, device, per_subject=1500, shuffle=True):
        self.X, self.Y, self.b, self.dev, self.k, self.sh = X, Y, batch, device, per_subject, shuffle
        self.groups = [rows[rec[rows] == r] for r in np.unique(rec[rows])] if shuffle else None
        self.rows = rows
    def _epoch_rows(self):
        if not self.sh: return self.rows
        pick = [g if len(g) <= self.k else np.random.choice(g, self.k, replace=False) for g in self.groups]
        return np.random.permutation(np.concatenate(pick))
    def __len__(self):
        n = sum(min(len(g), self.k) for g in self.groups) if self.sh else len(self.rows)
        return math.ceil(n / self.b)
    def __iter__(self):
        r = self._epoch_rows()
        for i in range(0, len(r), self.b):
            j = np.sort(r[i:i + self.b])
            x = torch.from_numpy(np.asarray(self.X[j], dtype=np.float32)).to(self.dev, non_blocking=True)
            y = {k: torch.from_numpy(v[j]).to(self.dev, non_blocking=True) for k, v in self.Y.items()}
            yield x, y, j

def augment(x):
    B = x.shape[0]; dev = x.device
    x = x * torch.empty(B, 2, 1, device=dev).uniform_(0.8, 1.2)
    x = x * torch.where(torch.rand(B, 2, 1, device=dev) < 0.5, -1.0, 1.0)
    x = x + 0.05 * torch.randn_like(x)
    L = x.shape[-1]; m = torch.rand(B, device=dev) < 0.3
    if m.any():
        s = torch.randint(0, L - 50, (B,), device=dev); w = torch.randint(10, 50, (B,), device=dev)
        t = torch.arange(L, device=dev)[None]
        mask = (t >= s[:, None]) & (t < (s + w)[:, None]) & m[:, None]
        x = x.masked_fill(mask[:, None, :], 0.0)
    drop = torch.rand(B, device=dev) < 0.1
    if drop.any():
        ch = torch.randint(0, 2, (B,), device=dev)
        keep = torch.ones(B, 2, 1, device=dev)
        keep[drop, ch[drop], 0] = 0.0
        x = x * keep
    return x

class StemBlock(nn.Module):
    def __init__(self, c_out=64):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(1, c_out, 7, 2, 3, bias=False), nn.BatchNorm1d(c_out), nn.ReLU(inplace=True),
                                 nn.MaxPool1d(3, 2, 1))
    def forward(self, x): return self.net(x)

class PhySEResidualBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=2, deriv_kernel=15, r=16, lam=1.0):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv1d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm1d(c_out), nn.ReLU(inplace=True),
            nn.Conv1d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm1d(c_out))
        self.deriv = nn.Sequential(
            nn.Conv1d(c_in, c_out, deriv_kernel, stride, deriv_kernel // 2, bias=False), nn.BatchNorm1d(c_out), nn.Hardtanh())
        self.lam = nn.Parameter(torch.tensor(float(lam)))
        h = max(c_out // r, 4)
        self.ub, self.ua = nn.Linear(c_out, h), nn.Linear(h, 2 * c_out)
        self.short = (nn.Identity() if (c_in == c_out and stride == 1) else
                      nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm1d(c_out)))
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        fm = self.main(x); fd = self.deriv(x) * self.lam
        a = self.ua(F.relu(self.ub(fm.mean(-1))))
        a = torch.softmax(a.view(-1, 2, fm.shape[1]), dim=1).unsqueeze(-1)
        return self.act(a[:, 0] * fm + a[:, 1] * fd + self.short(x))

class PhySEResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256, 512), **kw):
        super().__init__()
        self.stem = StemBlock(widths[0])
        c, st = widths[0], []
        for w in widths: st.append(PhySEResidualBlock(c, w, stride=2, **kw)); c = w
        self.stages = nn.ModuleList(st)
    def forward(self, x):
        x, feats = self.stem(x), []
        for s in self.stages: x = s(x); feats.append(x)
        return feats

class CrossModalFusion(nn.Module):
    def __init__(self, c_in, d=128, r=16):
        super().__init__()
        self.pa = nn.Sequential(nn.Conv1d(c_in, d, 1, bias=False), nn.BatchNorm1d(d))
        self.pb = nn.Sequential(nn.Conv1d(c_in, d, 1, bias=False), nn.BatchNorm1d(d))
        self.cat = nn.Conv1d(2 * d, d, 1)
        self.gate = nn.Sequential(nn.Flatten(), nn.Linear(2 * d, 2 * d // r), nn.ReLU(inplace=True), nn.Linear(2 * d // r, 3 * d))
        self.s = nn.Conv1d(d, 1, 1); self.off = nn.Conv1d(d, d, 3, 1, 1)
        self.enh = nn.Sequential(nn.Conv1d(d, d, 3, 1, 1, groups=d, bias=False), nn.BatchNorm1d(d), nn.ReLU(inplace=True))
        self.d = d
    def forward(self, fa, fb):
        a, b = self.pa(fa), self.pb(fb); c = torch.cat([a, b], 1)
        w = torch.softmax(self.gate(c.mean(-1, keepdim=True)).view(-1, 3, self.d), dim=1).unsqueeze(-1)
        f = w[:, 0] * (a + b) + w[:, 1] * (a * b) + w[:, 2] * self.cat(c)
        f = self.off(torch.sigmoid(self.s(f)) * f) + f
        return self.enh(f)

def make_sequences(idx, rows, L=12, max_gap=60.0, offset=0):
    sub = idx.iloc[rows][["record", "t"]].assign(row=rows).sort_values(["record", "t"])
    seqs = []
    for _, g in sub.groupby("record", sort=False):
        r, t = g.row.to_numpy(), g.t.to_numpy()
        breaks = np.where(np.diff(t) > max_gap)[0] + 1
        for seg in np.split(r, breaks):
            o = offset % L if len(seg) > L else 0
            cuts = list(range(o, len(seg), L)) if o else list(range(0, len(seg), L))
            if o: cuts = [0] + cuts
            for a, b in zip(cuts, cuts[1:] + [len(seg)]):
                chunk = seg[a:b]
                if len(chunk): seqs.append(np.pad(chunk, (0, L - len(chunk)), constant_values=-1))
    return np.array(seqs, dtype=np.int64).reshape(-1, L)

class SequenceBatches:
    def __init__(self, X, Y, idx, rows, batch_seq, device, L=12, per_subject=1500, train=True):
        self.X, self.Y, self.idx, self.rows, self.b, self.dev, self.L = X, Y, idx, rows, batch_seq, device, L
        self.k, self.train = per_subject // L, train
        self.rec = idx.record.to_numpy()
        self.fixed = None if train else make_sequences(idx, rows, L)
    def _epoch(self):
        if not self.train: return self.fixed
        s = make_sequences(self.idx, self.rows, self.L, offset=np.random.randint(self.L))
        own = self.rec[s[:, 0]]
        keep = np.concatenate([np.random.permutation(np.where(own == r)[0])[:self.k] for r in np.unique(own)])
        return s[np.random.permutation(keep)]
    def __len__(self):
        n = len(self.fixed) if not self.train else len(np.unique(self.rec[self.rows])) * self.k
        return math.ceil(n / self.b)
    def __iter__(self):
        S = self._epoch()
        for i in range(0, len(S), self.b):
            s = S[i:i + self.b]; m = s >= 0
            j = s[m]
            x = torch.from_numpy(np.asarray(self.X[j], dtype=np.float32)).to(self.dev, non_blocking=True)
            y = {k: torch.from_numpy(v[j]).to(self.dev, non_blocking=True) for k, v in self.Y.items()}
            yield x, torch.from_numpy(m).to(self.dev), y, j

class MLPHead(nn.Module):
    def __init__(self, d, ctx, out, hidden=64, p=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d + ctx, hidden), nn.ELU(), nn.Dropout(p), nn.Linear(hidden, out))
    def forward(self, f, c): return self.net(torch.cat([f.mean(-1), c], 1))

class MultiScalePredictor(nn.Module):
    def __init__(self, d, ctx, out, levels=3, p=0.3):
        super().__init__()
        self.heads = nn.ModuleList([MLPHead(d, ctx, out, p=p) for _ in range(levels)])
        self.alpha = nn.Parameter(torch.zeros(levels))
    def forward(self, feats, c):
        w = torch.softmax(self.alpha, 0)
        return sum(w[i] * h(f, c) for i, (h, f) in enumerate(zip(self.heads, feats)))

class TemporalContext(nn.Module):
    def __init__(self, d=128, L=12, layers=2, heads=4, p=0.1):
        super().__init__()
        self.proj = nn.Linear(3 * d, d); self.pos = nn.Parameter(torch.zeros(1, L, d))
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, p, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
    def forward(self, v, mask):
        h = self.proj(v) + self.pos[:, :v.shape[1]]
        return self.enc(h, src_key_padding_mask=~mask)

class PMRTTemporal(nn.Module):
    def __init__(self, d=128, L=12, deriv_kernel=15, r=16, p=0.3, feat_drop=0.1, stage_head=True, input_mode="both"):
        super().__init__()
        assert input_mode in ("both", "eeg", "emg")
        self.input_mode = input_mode
        self.branch_eeg = PhySEResNet(deriv_kernel=deriv_kernel, r=r)
        self.branch_emg = PhySEResNet(deriv_kernel=deriv_kernel, r=r)
        self.fusion = nn.ModuleList([CrossModalFusion(c, d, r) for c in (128, 256, 512)])
        self.fdrop = nn.Dropout1d(feat_drop)
        self.context = TemporalContext(d, L)
        outs = {"hr": 1, "rmssd": 1, "dis": len(DISORDERS)}
        if stage_head: outs["stg"] = len(STAGES)
        self.pred = nn.ModuleDict({k: MultiScalePredictor(d, d, o, p=p) for k, o in outs.items()})
    def windows(self, x):
        fa = self.branch_eeg(x[:, :1])[1:]; fb = self.branch_emg(x[:, 1:])[1:]
        m = [fu(a, b) for fu, a, b in zip(self.fusion, fa, fb)]
        p3 = m[2]
        p2 = m[1] + F.interpolate(p3, size=m[1].shape[-1], mode="nearest")
        p1 = m[0] + F.interpolate(p2, size=m[0].shape[-1], mode="nearest")
        return [self.fdrop(q) for q in (p1, p2, p3)]
    def forward(self, x, mask):
        if self.input_mode == "eeg": x = torch.cat([x[:, :1], torch.zeros_like(x[:, 1:])], 1)
        if self.input_mode == "emg": x = torch.cat([torch.zeros_like(x[:, :1]), x[:, 1:]], 1)
        fp = self.windows(x)
        v = torch.cat([q.mean(-1) for q in fp], 1)
        V = v.new_zeros(*mask.shape, v.shape[1]); V[mask] = v
        c = self.context(V, mask)[mask]
        out = {k: p(fp, c) for k, p in self.pred.items()}
        out["hr"], out["rmssd"] = out["hr"][:, 0], out["rmssd"][:, 0]
        return out

class Targets:
    def __init__(self, idx, train_rows):
        self.raw = idx[REG].to_numpy(np.float64)
        t = self._fwd(self.raw)
        self.mu, self.sd = t[train_rows].mean(0), t[train_rows].std(0)
        self.raw_sd = self.raw[train_rows].std(0)
    @staticmethod
    def _fwd(raw): return np.stack([raw[:, 0], np.log(np.maximum(raw[:, 1], 1.0))], 1)
    def encode(self): return ((self._fwd(self.raw) - self.mu) / self.sd).astype(np.float32)
    def decode(self, hr_z, rm_z):
        return hr_z * self.sd[0] + self.mu[0], np.exp(rm_z * self.sd[1] + self.mu[1])

class MultiTaskLoss:
    def __init__(self, idx_train, device, w_dis=0.5, w_stg=0.3, smoothing=0.1):
        subj = idx_train.drop_duplicates("record")
        n_dis = np.array([(subj.dis == i).sum() for i in range(len(DISORDERS))], float)
        w = 1.0 / np.maximum(n_dis, 1); w = w / w.sum() * len(w)
        n_stg = np.array([(idx_train.stg == i).sum() for i in range(len(STAGES))], float)
        ws = 1.0 / np.sqrt(np.maximum(n_stg, 1)); ws = ws / ws.sum() * len(ws)
        self.ce_dis = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device),
                                          ignore_index=IGNORE, label_smoothing=smoothing)
        self.ce_stg = nn.CrossEntropyLoss(weight=torch.tensor(ws, dtype=torch.float32, device=device),
                                          label_smoothing=smoothing)
        self.w_dis, self.w_stg = w_dis, w_stg
    def __call__(self, out, y):
        hub = lambda a, b: F.smooth_l1_loss(a.float(), b, beta=1.0)
        parts = {"hr": hub(out["hr"], y["hr"]), "rmssd": hub(out["rmssd"], y["rmssd"])}
        if (y["dis"] != IGNORE).any(): parts["dis"] = self.ce_dis(out["dis"].float(), y["dis"])
        else: parts["dis"] = torch.zeros((), device=y["hr"].device)
        total = parts["hr"] + parts["rmssd"] + self.w_dis * parts["dis"]
        if "stg" in out:
            parts["stg"] = self.ce_stg(out["stg"].float(), y["stg"]); total = total + self.w_stg * parts["stg"]
        return total, parts

class EMA:
    def __init__(self, model, decay=0.999):
        import copy
        self.m = copy.deepcopy(model).eval(); self.d = decay
        for p in self.m.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for e, p in zip(self.m.parameters(), model.parameters()): e.mul_(self.d).add_(p.detach(), alpha=1 - self.d)
        for e, b in zip(self.m.buffers(), model.buffers()): e.copy_(b)

@torch.no_grad()
def predict(model, loader):
    model.eval(); res, J = {}, []
    for x, mask, _, j in loader:
        with torch.autocast(x.device.type, dtype=torch.float16, enabled=x.device.type == "cuda"):
            out = model(x, mask)
        for k, v in out.items():
            v = v.float()
            res.setdefault(k, []).append((torch.softmax(v, -1) if v.ndim == 2 else v).cpu().numpy())
        J.append(j)
    J = np.concatenate(J); o = np.argsort(J, kind="stable")
    return J[o], {k: np.concatenate(v)[o] for k, v in res.items()}

def reg_metrics(y, p):
    e = p - y
    return {"MAE": float(np.mean(np.abs(e))), "RMSE": float(np.sqrt(np.mean(e ** 2))), "ME": float(np.mean(e)),
            "SD": float(np.std(e)), "r": float(np.corrcoef(y, p)[0, 1]),
            "R2": float(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2))}

def val_scores(p, j, tg, dis, stg, rec):
    hr, rm = tg.decode(p["hr"], p["rmssd"])
    hr_mae = float(np.mean(np.abs(hr - tg.raw[j, 0]))); rm_mae = float(np.mean(np.abs(rm - tg.raw[j, 1])))
    s = {"stress_score": hr_mae / tg.raw_sd[0] + rm_mae / tg.raw_sd[1], "hr_MAE": hr_mae, "rmssd_MAE": rm_mae}
    d = pd.DataFrame(p["dis"], columns=DISORDERS); d["rec"] = rec[j]; d["y"] = dis[j]
    d = d[d.y != IGNORE].groupby("rec").agg({**{c: "mean" for c in DISORDERS}, "y": "first"})
    if len(d):
        pr = d[DISORDERS].to_numpy()
        s["dis_subject_loss"] = float(-np.mean(np.log(pr[np.arange(len(d)), d.y.to_numpy()] + 1e-8)))
        s["dis_subject_acc"] = float(np.mean(pr.argmax(1) == d.y.to_numpy()))
    else:
        s["dis_subject_loss"], s["dis_subject_acc"] = float("nan"), float("nan")
    if "stg" in p: s["stage_acc"] = float(np.mean(p["stg"].argmax(1) == stg[j]))
    return s

def train_fold(model, tr, va, loss_fn, tg, dis, stg, rec, run_dir, epochs=60, lr=5e-4, wd=0.05,
               patience=12, min_epochs=15, use_aug=True, ema_decay=0.999):
    run = Path(run_dir); run.mkdir(parents=True, exist_ok=True)
    dev = next(model.parameters()).device
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs, steps_per_epoch=len(tr),
                                                pct_start=max(0.1, 3 / (epochs * len(tr))))
    scaler = torch.amp.GradScaler(enabled=dev.type == "cuda")
    ema = EMA(model, ema_decay)
    state = {"epoch": 0, "best_stress": float("inf"), "best_dis": float("inf"), "bad": 0, "history": []}
    if (run / "last.pt").exists():
        ck = torch.load(run / "last.pt", map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"]); ema.m.load_state_dict(ck["ema"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"]); state = ck["state"]
        print(f"  resumed after epoch {state['epoch']}")
    while state["epoch"] < epochs and not (state["epoch"] >= min_epochs and state["bad"] >= patience):
        ep = state["epoch"] + 1; t0 = time.time(); model.train(); tot, nb = 0.0, 0
        for step, (x, mask, y, _) in enumerate(tr):
            if step >= len(tr): break
            if use_aug: x = augment(x)
            with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
                out = model(x, mask)
            loss, _ = loss_fn(out, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
            tot += loss.item(); nb += 1
        j, p = predict(ema.m, va)
        s = val_scores(p, j, tg, dis, stg, rec)
        rec_h = {"epoch": ep, "train_loss": tot / nb, **{f"val_{k}": v for k, v in s.items()},
                 "lr": opt.param_groups[0]["lr"], "sec": time.time() - t0}
        state["history"].append(rec_h); state["epoch"] = ep
        flag, improved = "", False
        if s["stress_score"] < state["best_stress"]:
            state["best_stress"] = s["stress_score"]; improved = True; flag += " *stress*"
            torch.save({"model": ema.m.state_dict(), "epoch": ep}, run / "best_stress.pt")
        if s["dis_subject_loss"] < state["best_dis"]:
            state["best_dis"] = s["dis_subject_loss"]; improved = True; flag += " *dis*"
            torch.save({"model": ema.m.state_dict(), "epoch": ep}, run / "best_dis.pt")
        state["bad"] = 0 if improved else state["bad"] + 1
        torch.save({"model": model.state_dict(), "ema": ema.m.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "scaler": scaler.state_dict(), "state": state}, run / "last.pt")
        pd.DataFrame(state["history"]).to_csv(run / "history.csv", index=False)
        print(f"  ep {ep:3d} | train {rec_h['train_loss']:.3f} | stress score {s['stress_score']:.3f} | HR MAE {s['hr_MAE']:.2f} | "
              f"RMSSD MAE {s['rmssd_MAE']:.1f} | dis subj acc {s['dis_subject_acc']:.2f} loss {s['dis_subject_loss']:.2f}" +
              (f" | stage acc {s['stage_acc']:.2f}" if "stage_acc" in s else "") + f" | {rec_h['sec']:.0f}s{flag}", flush=True)
    h = pd.DataFrame(state["history"])
    print(f"  finished after {state['epoch']} epochs | best stress epoch {int(h.loc[h.val_stress_score.idxmin(), 'epoch'])} "
          f"| best disorder epoch {int(h.loc[h.val_dis_subject_loss.idxmin(), 'epoch'])}")


Writing /kaggle/working/pmrt_mt3.py


## 4. Load module

In [ ]:
# Load the module
sys.path.insert(0, ROOT)
import pmrt_mt3 as M
importlib.reload(M)
print("pmrt_mt3", M.VERSION, "loaded")

pmrt_mt3 mt3 loaded


## 5. Data and folds

In [ ]:
# Data and folds (same folds as all previous runs)
X, idx = M.load_data(str(DATA))
REC = idx.record.to_numpy()
FOLD = json.loads((DATA / "folds_5.json").read_text())
print("windows:", X.shape, "| subjects:", idx.record.nunique())

windows: (483170, 2, 625) | subjects: 103


## 6. Training function

In [ ]:
# Training function: one fold of one ablation per call (resumable)
DIS, STG = idx.dis.to_numpy(np.int64), idx.stg.to_numpy(np.int64)

def run_fold(mode, k):
    assert mode in ("eeg", "emg") and k in range(N_FOLDS)
    run = Path(ROOT) / "runs" / f"pmrt_mt3_{mode}only_cv5"
    run.mkdir(parents=True, exist_ok=True)
    (run / "config.json").write_text(json.dumps({"N_FOLDS": N_FOLDS, "SEQ_LEN": SEQ_LEN, "EPOCHS": EPOCHS, "PATIENCE": PATIENCE,
        "MIN_EPOCHS": MIN_EPOCHS, "BATCH_SEQ": BATCH_SEQ, "LR": LR, "WD": WD, "PER_SUBJECT": PER_SUBJECT, "USE_AUG": USE_AUG,
        "USE_STAGE_HEAD": USE_STAGE_HEAD, "EMA_DECAY": EMA_DECAY, "SEED": SEED, "INPUT_MODE": mode, "version": M.VERSION}, indent=1))
    fdir = run / f"fold{k}"
    if (fdir / "test_pred.csv").exists():
        print(f"{mode} only, fold {k}: already finished - skipped"); return
    torch.manual_seed(SEED + k); np.random.seed(SEED + k)
    rows = M.fold_rows(idx, FOLD, k, N_FOLDS)
    tg = M.Targets(idx, rows["train"])
    yz = tg.encode()
    Y = {"hr": yz[:, 0], "rmssd": yz[:, 1], "dis": DIS, "stg": STG}
    tr = M.SequenceBatches(X, Y, idx, rows["train"], BATCH_SEQ, DEV, SEQ_LEN, PER_SUBJECT, train=True)
    va = M.SequenceBatches(X, Y, idx, rows["val"], 64, DEV, SEQ_LEN, train=False)
    te = M.SequenceBatches(X, Y, idx, rows["test"], 64, DEV, SEQ_LEN, train=False)
    print(f"===== {mode} only | fold {k} | train {len(np.unique(REC[rows['train']]))} subj | "
          f"val {len(np.unique(REC[rows['val']]))} | test {len(np.unique(REC[rows['test']]))} =====")
    loss_fn = M.MultiTaskLoss(idx.iloc[rows["train"]], DEV)
    model = M.PMRTTemporal(L=SEQ_LEN, stage_head=USE_STAGE_HEAD, input_mode=mode).to(DEV)
    M.train_fold(model, tr, va, loss_fn, tg, DIS, STG, REC, fdir, epochs=EPOCHS, lr=LR, wd=WD,
                 patience=PATIENCE, min_epochs=MIN_EPOCHS, use_aug=USE_AUG, ema_decay=EMA_DECAY)
    model.load_state_dict(torch.load(fdir / "best_stress.pt", map_location=DEV, weights_only=False)["model"])
    j, p = M.predict(model, te)
    model.load_state_dict(torch.load(fdir / "best_dis.pt", map_location=DEV, weights_only=False)["model"])
    j2, p2 = M.predict(model, te)
    assert (j == j2).all() and (j == np.sort(rows["test"])).all()
    out = idx.iloc[j][["record", "group", "stage", "clipped", "dis", "stg"] + M.REG].reset_index(drop=True)
    out["fold"] = k
    out["hr_pred"], out["rmssd_pred"] = tg.decode(p["hr"], p["rmssd"])
    for c, name in enumerate(M.DISORDERS): out[f"p_{name}"] = p2["dis"][:, c]
    for c, name in enumerate(M.STAGES): out[f"p_{name}"] = p["stg"][:, c]
    out["mu_hr"], out["mu_rmssd"] = np.median(tg.raw[rows["train"]], 0)
    out.to_csv(fdir / "test_pred.csv", index=False)
    done = [i for i in range(N_FOLDS) if (run / f"fold{i}" / "test_pred.csv").exists()]
    print(f"{mode} only, fold {k} saved | finished folds: {done}")

print("run_fold(mode, k) ready")

run_fold(mode, k) ready


## 7. Train the folds in TODO

In [ ]:
# Train the folds in TODO (no new fold after START_LIMIT_H)
for mode, k in TODO:
    if (Path(ROOT) / "runs" / f"pmrt_mt3_{mode}only_cv5" / f"fold{k}" / "test_pred.csv").exists():
        print(f"{mode} only, fold {k}: already finished - skipped"); continue
    used = (time.time() - T_START) / 3600
    if used > START_LIMIT_H:
        print(f"\n{used:.1f} h used - no new fold started; continue in the next run"); break
    run_fold(mode, k)
for rn in sorted((Path(ROOT) / "runs").glob("pmrt_mt3_*only_cv5")):
    for f in rn.glob("fold*/last.pt"):
        if (f.parent / "test_pred.csv").exists(): f.unlink()

emg only, fold 0: already finished - skipped
eeg only, fold 1: already finished - skipped
===== emg only | fold 1 | train 61 subj | val 21 | test 21 =====
  ep   1 | train 2.175 | stress score 1.769 | HR MAE 7.68 | RMSSD MAE 33.2 | dis subj acc 0.30 loss 1.77 | stage acc 0.46 | 184s *stress* *dis*
  ep   2 | train 1.990 | stress score 1.743 | HR MAE 7.58 | RMSSD MAE 32.7 | dis subj acc 0.20 loss 1.83 | stage acc 0.43 | 85s *stress*
  ep   3 | train 1.866 | stress score 1.697 | HR MAE 7.36 | RMSSD MAE 31.9 | dis subj acc 0.20 loss 1.82 | stage acc 0.44 | 92s *stress*
  ep   4 | train 1.801 | stress score 1.722 | HR MAE 7.29 | RMSSD MAE 33.0 | dis subj acc 0.20 loss 1.86 | stage acc 0.46 | 83s
  ep   5 | train 1.750 | stress score 1.678 | HR MAE 6.83 | RMSSD MAE 33.1 | dis subj acc 0.20 loss 1.86 | stage acc 0.45 | 83s *stress*
  ep   6 | train 1.684 | stress score 1.645 | HR MAE 6.59 | RMSSD MAE 32.8 | dis subj acc 0.20 loss 1.92 | stage acc 0.44 | 87s *stress*
  ep   7 | train 1.637 | 

## 8. Comparison: fusion vs EEG only vs EMG only (folds finished by all models)

In [ ]:
# Fusion vs single modality on identical subjects
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, cohen_kappa_score
runs = {"EEG + EMG (fusion)": "pmrt_mt3_cv5", "EEG only": "pmrt_mt3_eegonly_cv5", "EMG only": "pmrt_mt3_emgonly_cv5"}
fin = {n: {k for k in range(N_FOLDS) if (Path(ROOT) / "runs" / rn / f"fold{k}" / "test_pred.csv").exists()} for n, rn in runs.items()}
runs = {n: rn for n, rn in runs.items() if fin[n]}
common = sorted(set.intersection(*[fin[n] for n in runs]))
print("finished folds:", {n: sorted(f) for n, f in fin.items()}, "| compared on folds:", common)
pcols = [f"p_{c}" for c in M.DISORDERS]
def zin(df, col):
    g = df.groupby("record")[col]; return (df[col] - g.transform("mean")) / g.transform("std")
rows, per_subj = [], {}
for name, rn in runs.items():
    d = pd.concat([pd.read_csv(Path(ROOT) / "runs" / rn / f"fold{k}" / "test_pred.csv") for k in common], ignore_index=True)
    hr, rm = M.reg_metrics(d.hr.to_numpy(), d.hr_pred.to_numpy()), M.reg_metrics(d.rmssd.to_numpy(), d.rmssd_pred.to_numpy())
    y2 = ((zin(d, "hr") - zin(d, "rmssd")) > 1).astype(int); s2 = zin(d, "hr_pred") - zin(d, "rmssd_pred")
    y1 = (zin(d, "hr") > 1).astype(int); s1 = zin(d, "hr_pred")
    S = d.groupby("record").agg(dis=("dis", "first"), group=("group", "first"), **{c: (c, "mean") for c in pcols})
    s6 = S[S.dis != M.IGNORE]
    sp = d[[f"p_{s}" for s in M.STAGES]].to_numpy().argmax(1)
    rows.append({"model": name, "subjects": d.record.nunique(), "HR MAE": hr["MAE"], "HR r": hr["r"], "RMSSD r": rm["r"],
                 "stress AUC": roc_auc_score(y2, s2), "stress bal.acc": balanced_accuracy_score(y2, (s2 > 1).astype(int)),
                 "stress AUC (HR only)": roc_auc_score(y1, s1),
                 "disorder bal.acc": balanced_accuracy_score(s6.dis, s6[pcols].to_numpy().argmax(1)),
                 "healthy vs dis. AUC": roc_auc_score((S.group != "Healthy").astype(int), 1 - S.p_Healthy)
                                        if S.group.eq("Healthy").any() and S.group.ne("Healthy").any() else np.nan,
                 "stage kappa": cohen_kappa_score(d.stg, sp)})
    per_subj[name] = d.assign(e=(d.hr_pred - d.hr).abs()).groupby("record").e.mean()
print(pd.DataFrame(rows).set_index("model").round(3).to_string())
ref = "EEG + EMG (fusion)"
for name in per_subj:
    if name == ref or ref not in per_subj: continue
    a, b = per_subj[ref].align(per_subj[name], join="inner")
    print(f"\nper-subject HR MAE, fusion vs {name}: fusion better in {int((a < b).sum())}/{len(a)} subjects | "
          f"median diff {np.median(a - b):+.2f} bpm | Wilcoxon p = {wilcoxon(a, b).pvalue:.2g}")

finished folds: {'EEG + EMG (fusion)': [0, 1, 2, 3, 4], 'EEG only': [0, 1, 2, 3, 4], 'EMG only': [0, 1, 2, 3, 4]} | compared on folds: [0, 1, 2, 3, 4]
                    subjects  HR MAE   HR r  RMSSD r  stress AUC  stress bal.acc  stress AUC (HR only)  disorder bal.acc  healthy vs dis. AUC  stage kappa
model                                                                                                                                                     
EEG + EMG (fusion)       103   4.300  0.730    0.205       0.719           0.662                 0.796             0.378                0.815        0.626
EEG only                 103   6.502  0.437    0.161       0.640           0.596                 0.676             0.286                0.679        0.568
EMG only                 103   4.215  0.706    0.169       0.716           0.661                 0.802             0.320                0.792        0.222

per-subject HR MAE, fusion vs EEG only: fusion better in 87/103 subjects 